In [1]:
# ============================================================
# Regression 3-Source Evaluation Notebook
# ============================================================

import numpy as np
import pandas as pd
import torch
from torch import nn
import os
import joblib
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

# Plot style
plt.style.use("default")
sns.set_theme(style="whitegrid")

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

print("Device:", DEVICE)

# Global config for evaluation
CONFIG = {
    "DATA_DIR": "../data/regression_3src",
    "TRAINING_DIR": "../models/regression_3src",
    "THRESHOLDS": [0.01, 0.02, 0.03, 0.04, 0.05],
    "NOISE_LEVELS": [0.0, 0.01, 0.03, 0.05, 0.10],
    "NUM_ANGLES": 30,
}

print("CONFIG loaded.")


Device: cpu
CONFIG loaded.


In [2]:
# ============================================================
# Cell 2 — Load final Stage‑8 model + scalers
# ============================================================

from inverse_source_em.training.model_3src import ThreeSourceMultiHeadBig

# Paths (relative to notebooks/)
MODEL_PATH = "../models/regression_3src/best_model_stage_8.pt"
SCALER_X_PATH = "../data/regression_3src/stage_8_scaler_X.pkl"
SCALER_Y_PATH = "../data/regression_3src/stage_8_scaler_y.pkl"

# Input dimension for 3‑source regression
INPUT_DIM = 4 * CONFIG["NUM_ANGLES"]   # 4 × 30 = 120

# Instantiate model
model = ThreeSourceMultiHeadBig(INPUT_DIM).to(DEVICE)

# Load weights
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("Loaded final Stage‑8 model:")
print("  ", MODEL_PATH)

# Load scalers
scaler_X = joblib.load(SCALER_X_PATH)
scaler_y = joblib.load(SCALER_Y_PATH)

print("Loaded Stage‑8 scalers:")
print("  ", SCALER_X_PATH)
print("  ", SCALER_Y_PATH)


Loaded final Stage‑8 model:
   ../models/regression_3src/best_model_stage_8.pt
Loaded Stage‑8 scalers:
   ../data/regression_3src/stage_8_scaler_X.pkl
   ../data/regression_3src/stage_8_scaler_y.pkl


In [3]:
# ============================================================
# Cell 3 — Evaluation dataset generator (clean, 3 sources)
# ============================================================

from tqdm import tqdm

# Geometry curriculum levels (same as training)
GEOMETRY_LEVELS = {
    1: {"dr_min": 0.10, "dphi_min_deg": 40},
    2: {"dr_min": 0.08, "dphi_min_deg": 25},
    3: {"dr_min": 0.08, "dphi_min_deg": 20},
    4: {"dr_min": 0.07, "dphi_min_deg": 15},
    5: {"dr_min": 0.06, "dphi_min_deg": 10},
    6: {"dr_min": 0.05, "dphi_min_deg": 8},
    7: {"dr_min": 0.05, "dphi_min_deg": 6},
    8: {"dr_min": 0.05, "dphi_min_deg": 5},
}

NUM_ANGLES = CONFIG["NUM_ANGLES"]
theta = np.linspace(0, 2*np.pi, NUM_ANGLES, endpoint=False)

# ------------------------------------------------------------
# Canonical ordering for 3 sources
# ------------------------------------------------------------
def canonical_order_three(rho1, phi1, rho2, phi2, rho3, phi3):
    pts = [(rho1, phi1), (rho2, phi2), (rho3, phi3)]
    pts_sorted = sorted(pts, key=lambda t: (t[0], t[1]))
    (rho1, phi1), (rho2, phi2), (rho3, phi3) = pts_sorted
    return rho1, phi1, rho2, phi2, rho3, phi3

# ------------------------------------------------------------
# Angular difference helper
# ------------------------------------------------------------
def ang_diff(a, b):
    return np.abs(((a - b + np.pi) % (2*np.pi)) - np.pi)

# ------------------------------------------------------------
# Surrogate-based forward model (E + H)
# ------------------------------------------------------------
from inverse_source_em.surrogate.surrogate import SurrogateEM
from inverse_source_em.surrogate.surrogate_wrapper import SurrogateWrapper
from inverse_source_em.physics.physics_tm import PhysicsTM

# Physics radius
phys = PhysicsTM()
R = phys.R

# Surrogate model paths (relative to notebooks/)
PATH_E = "../models/surrogate_Esurf.pth"
PATH_H = "../models/surrogate_Hsurf.pth"

# Instantiate surrogate
sur = SurrogateEM(path_E=PATH_E, path_H=PATH_H, R=R)
sur_wrap = SurrogateWrapper(sur)

print("Surrogate wrapper ready.")

# ------------------------------------------------------------
# Forward model for 3 sources
# ------------------------------------------------------------
def get_features_three_sources(rho1, phi1, rho2, phi2, rho3, phi3):
    E1 = sur_wrap.Esurf(rho1, phi1, theta)
    E2 = sur_wrap.Esurf(rho2, phi2, theta)
    E3 = sur_wrap.Esurf(rho3, phi3, theta)

    H1 = sur_wrap.Hsurf(rho1, phi1, theta)
    H2 = sur_wrap.Hsurf(rho2, phi2, theta)
    H3 = sur_wrap.Hsurf(rho3, phi3, theta)

    E_total = E1 + E2 + E3
    H_total = H1 + H2 + H3

    Ere = np.real(E_total)
    Eim = np.imag(E_total)
    Hre = np.real(H_total)
    Him = np.imag(H_total)

    return np.concatenate([Ere, Eim, Hre, Him], axis=0).astype(np.float64)

# ------------------------------------------------------------
# Evaluation dataset generator
# ------------------------------------------------------------
def generate_eval_dataset(num_samples, geom_level):
    """
    Clean evaluation dataset (no noise).
    Same geometry constraints as training.
    """
    params = GEOMETRY_LEVELS[geom_level]
    dr_min = params["dr_min"]
    dphi_min = np.deg2rad(params["dphi_min_deg"])

    X_list = []
    Y_list = []

    pbar = tqdm(total=num_samples, desc=f"Eval Level {geom_level}")

    while len(X_list) < num_samples:

        rho = np.random.uniform(0.1, 0.9, size=3)
        phi = np.random.uniform(-np.pi, np.pi, size=3)

        ok = True
        for i in range(3):
            for j in range(i+1, 3):
                if abs(rho[i] - rho[j]) < dr_min:
                    ok = False
                if ang_diff(phi[i], phi[j]) < dphi_min:
                    ok = False
        if not ok:
            continue

        rho1, phi1, rho2, phi2, rho3, phi3 = canonical_order_three(
            rho[0], phi[0], rho[1], phi[1], rho[2], phi[2]
        )

        feats = get_features_three_sources(rho1, phi1, rho2, phi2, rho3, phi3)

        y_vec = [
            rho1*np.cos(phi1), rho1*np.sin(phi1),
            rho2*np.cos(phi2), rho2*np.sin(phi2),
            rho3*np.cos(phi3), rho3*np.sin(phi3)
        ]

        X_list.append(feats)
        Y_list.append(y_vec)
        pbar.update(1)

    pbar.close()

    return np.array(X_list, dtype=np.float64), np.array(Y_list, dtype=np.float64)

print("Evaluation dataset generator ready.")


Surrogate wrapper ready.
Evaluation dataset generator ready.


In [4]:
# ============================================================
# Cell 4 — Clean evaluation (no noise)
# ============================================================

from sklearn.metrics import r2_score

# ------------------------------------------------------------
# 1. Generate evaluation dataset
# ------------------------------------------------------------

N_EVAL = 5000
EVAL_STAGE = 8   # hardest geometry

print(f"Generating clean evaluation dataset (N={N_EVAL}, stage={EVAL_STAGE})...")
X_eval_raw, y_eval_raw = generate_eval_dataset(N_EVAL, EVAL_STAGE)

print("Eval dataset shapes:")
print("  X_eval_raw:", X_eval_raw.shape)
print("  y_eval_raw:", y_eval_raw.shape)

# ------------------------------------------------------------
# 2. Scale inputs (same scaler_X as Stage 8)
# ------------------------------------------------------------

X_eval = scaler_X.transform(X_eval_raw)

# ------------------------------------------------------------
# 3. Convert to torch
# ------------------------------------------------------------

X_eval_t = torch.tensor(X_eval, dtype=torch.float64).to(DEVICE)

# ------------------------------------------------------------
# 4. Forward pass
# ------------------------------------------------------------

model.eval()
with torch.no_grad():
    rho_pred, cos_pred, sin_pred = model(X_eval_t)

rho_pred = rho_pred.cpu().numpy()
cos_pred = cos_pred.cpu().numpy()
sin_pred = sin_pred.cpu().numpy()

# ------------------------------------------------------------
# 5. Convert predictions to Cartesian
# ------------------------------------------------------------

phi_pred = np.arctan2(sin_pred, cos_pred)

x_pred = rho_pred * np.cos(phi_pred)
y_pred = rho_pred * np.sin(phi_pred)

y_pred_cart = np.column_stack([
    x_pred[:,0], y_pred[:,0],
    x_pred[:,1], y_pred[:,1],
    x_pred[:,2], y_pred[:,2]
])

print("\nClean forward pass complete.")
print("y_pred_cart shape:", y_pred_cart.shape)


Generating clean evaluation dataset (N=5000, stage=8)...


Eval Level 8: 100%|██████████| 5000/5000 [00:08<00:00, 602.75it/s]


Eval dataset shapes:
  X_eval_raw: (5000, 120)
  y_eval_raw: (5000, 6)

Clean forward pass complete.
y_pred_cart shape: (5000, 6)


In [5]:
# ============================================================
# Cell 5 — Clean evaluation metrics (R² per source + MAE)
# ============================================================

# True Cartesian coordinates
xA_true, yA_true = y_eval_raw[:, 0], y_eval_raw[:, 1]
xB_true, yB_true = y_eval_raw[:, 2], y_eval_raw[:, 3]
xC_true, yC_true = y_eval_raw[:, 4], y_eval_raw[:, 5]

# Predicted Cartesian coordinates
xA_pred, yA_pred = y_pred_cart[:, 0], y_pred_cart[:, 1]
xB_pred, yB_pred = y_pred_cart[:, 2], y_pred_cart[:, 3]
xC_pred, yC_pred = y_pred_cart[:, 4], y_pred_cart[:, 5]

# ------------------------------------------------------------
# 1. Compute R² per source
# ------------------------------------------------------------
R2_A = r2_score(np.column_stack([xA_true, yA_true]),
                np.column_stack([xA_pred, yA_pred]))

R2_B = r2_score(np.column_stack([xB_true, yB_true]),
                np.column_stack([xB_pred, yB_pred]))

R2_C = r2_score(np.column_stack([xC_true, yC_true]),
                np.column_stack([xC_pred, yC_pred]))

# ------------------------------------------------------------
# 2. Mean Absolute Error (Cartesian)
# ------------------------------------------------------------
MAE = np.mean(np.abs(y_pred_cart - y_eval_raw))

# ------------------------------------------------------------
# 3. Print results
# ------------------------------------------------------------
print("\n================ CLEAN EVALUATION RESULTS ================")
print(f"R² Source A: {R2_A:.4f}")
print(f"R² Source B: {R2_B:.4f}")
print(f"R² Source C: {R2_C:.4f}")
print(f"Mean Absolute Error (Cartesian): {MAE:.4f}")
print("==========================================================")



================ CLEAN EVALUATION RESULTS ================
R² Source A: 0.9873
R² Source B: 0.9881
R² Source C: 0.9949
Mean Absolute Error (Cartesian): 0.0171


In [6]:
# ============================================================
# Cell 6 — Noise-sweep evaluation (0% → 10%)
# ============================================================

NOISE_LEVELS = CONFIG["NOISE_LEVELS"]
results = []

print("\nRunning noise-sweep evaluation...\n")

for noise_frac in NOISE_LEVELS:

    print(f"--- Noise level: {int(noise_frac*100)}% ---")

    # --------------------------------------------------------
    # 1. Copy raw evaluation features
    # --------------------------------------------------------
    X_noisy = X_eval_raw.copy()

    if noise_frac > 0:
        num_angles = CONFIG["NUM_ANGLES"]
        feats_per_angle = X_noisy.shape[1] // num_angles

        # reshape to [N, 30, feats]
        X_noisy = X_noisy.reshape(N_EVAL, num_angles, feats_per_angle)

        k = int(noise_frac * num_angles)

        for i in range(N_EVAL):
            idx = np.random.permutation(num_angles)[:k]
            noise = 0.01 * np.random.randn(len(idx), feats_per_angle)
            X_noisy[i, idx] += noise

        # flatten back
        X_noisy = X_noisy.reshape(N_EVAL, num_angles * feats_per_angle)

    # --------------------------------------------------------
    # 2. Scale noisy inputs
    # --------------------------------------------------------
    X_noisy_scaled = scaler_X.transform(X_noisy)

    # --------------------------------------------------------
    # 3. Forward pass
    # --------------------------------------------------------
    X_t = torch.tensor(X_noisy_scaled, dtype=torch.float64).to(DEVICE)

    with torch.no_grad():
        rho_p, cos_p, sin_p = model(X_t)

    rho_p = rho_p.cpu().numpy()
    cos_p = cos_p.cpu().numpy()
    sin_p = sin_p.cpu().numpy()

    # --------------------------------------------------------
    # 4. Convert to Cartesian
    # --------------------------------------------------------
    phi_p = np.arctan2(sin_p, cos_p)

    x_p = rho_p * np.cos(phi_p)
    y_p = rho_p * np.sin(phi_p)

    y_pred_cart_noise = np.column_stack([
        x_p[:,0], y_p[:,0],
        x_p[:,1], y_p[:,1],
        x_p[:,2], y_p[:,2]
    ])

    # --------------------------------------------------------
    # 5. Compute R² per source
    # --------------------------------------------------------
    R2_A = r2_score(np.column_stack([xA_true, yA_true]),
                    np.column_stack([y_pred_cart_noise[:,0], y_pred_cart_noise[:,1]]))

    R2_B = r2_score(np.column_stack([xB_true, yB_true]),
                    np.column_stack([y_pred_cart_noise[:,2], y_pred_cart_noise[:,3]]))

    R2_C = r2_score(np.column_stack([xC_true, yC_true]),
                    np.column_stack([y_pred_cart_noise[:,4], y_pred_cart_noise[:,5]]))

    MAE = np.mean(np.abs(y_pred_cart_noise - y_eval_raw))

    results.append([noise_frac, R2_A, R2_B, R2_C, MAE])

    print(f"R² A={R2_A:.4f}, B={R2_B:.4f}, C={R2_C:.4f}, MAE={MAE:.4f}")

# ------------------------------------------------------------
# 6. Convert results to array for later plotting
# ------------------------------------------------------------
results = np.array(results)

print("\n================ NOISE SWEEP COMPLETE ================")
print("Columns: [noise_frac, R2_A, R2_B, R2_C, MAE]")
print(results)
print("======================================================")



Running noise-sweep evaluation...

--- Noise level: 0% ---
R² A=0.9873, B=0.9881, C=0.9949, MAE=0.0171
--- Noise level: 1% ---
R² A=0.9873, B=0.9881, C=0.9949, MAE=0.0171
--- Noise level: 3% ---
R² A=0.9873, B=0.9881, C=0.9949, MAE=0.0171
--- Noise level: 5% ---
R² A=0.9873, B=0.9881, C=0.9949, MAE=0.0171
--- Noise level: 10% ---
R² A=0.9871, B=0.9878, C=0.9949, MAE=0.0171

================ NOISE SWEEP COMPLETE ================
Columns: [noise_frac, R2_A, R2_B, R2_C, MAE]
[[0.         0.98731461 0.98813124 0.99493794 0.0171016 ]
 [0.01       0.98731461 0.98813124 0.99493794 0.0171016 ]
 [0.03       0.98731461 0.98813124 0.99493794 0.0171016 ]
 [0.05       0.98727702 0.98805794 0.99494071 0.01710836]
 [0.1        0.98714731 0.98778974 0.99487788 0.01714547]]


In [7]:
# ============================================================
# Cell 7 — Percentile analysis (clean + noisy)
# ============================================================

percentiles = [50, 75, 90, 95, 99]

print("\n================ PERCENTILE ANALYSIS ================\n")

# ------------------------------------------------------------
# 1. Clean (noise = 0.0)
# ------------------------------------------------------------

print("--- Noise level: 0% ---")

# Cartesian errors per sample (clean predictions from Cell 4)
err_clean = np.linalg.norm(y_pred_cart - y_eval_raw, axis=1)

pvals_clean = np.percentile(err_clean, percentiles)

for p, v in zip(percentiles, pvals_clean):
    print(f"  {p}th percentile error: {v:.4f}")

print(f"  Mean error: {np.mean(err_clean):.4f}")
print(f"  Max error : {np.max(err_clean):.4f}\n")

# ------------------------------------------------------------
# 2. Noisy levels (from Cell 6)
# ------------------------------------------------------------

for noise_frac, R2_A, R2_B, R2_C, MAE in results:

    print(f"--- Noise level: {int(noise_frac*100)}% ---")

    # Recompute Cartesian errors for this noise level
    # (We must regenerate predictions because y_pred_cart_noise was overwritten)
    num_angles = CONFIG["NUM_ANGLES"]
    feats_per_angle = X_eval_raw.shape[1] // num_angles

    X_noisy = X_eval_raw.copy()

    if noise_frac > 0:
        X_noisy = X_noisy.reshape(N_EVAL, num_angles, feats_per_angle)
        k = int(noise_frac * num_angles)

        for i in range(N_EVAL):
            idx = np.random.permutation(num_angles)[:k]
            noise = 0.01 * np.random.randn(len(idx), feats_per_angle)
            X_noisy[i, idx] += noise

        X_noisy = X_noisy.reshape(N_EVAL, num_angles * feats_per_angle)

    # Scale
    X_noisy_scaled = scaler_X.transform(X_noisy)
    X_t = torch.tensor(X_noisy_scaled, dtype=torch.float64).to(DEVICE)

    # Forward pass
    with torch.no_grad():
        rho_p, cos_p, sin_p = model(X_t)

    rho_p = rho_p.cpu().numpy()
    cos_p = cos_p.cpu().numpy()
    sin_p = sin_p.cpu().numpy()

    phi_p = np.arctan2(sin_p, cos_p)
    x_p = rho_p * np.cos(phi_p)
    y_p = rho_p * np.sin(phi_p)

    y_pred_cart_noise = np.column_stack([
        x_p[:,0], y_p[:,0],
        x_p[:,1], y_p[:,1],
        x_p[:,2], y_p[:,2]
    ])

    # Cartesian error
    err = np.linalg.norm(y_pred_cart_noise - y_eval_raw, axis=1)

    # Percentiles
    pvals = np.percentile(err, percentiles)

    for p, v in zip(percentiles, pvals):
        print(f"  {p}th percentile error: {v:.4f}")

    print(f"  Mean error: {np.mean(err):.4f}")
    print(f"  Max error : {np.max(err):.4f}\n")

print("======================================================")



================ PERCENTILE ANALYSIS ================

--- Noise level: 0% ---
  50th percentile error: 0.0395
  75th percentile error: 0.0556
  90th percentile error: 0.0778
  95th percentile error: 0.1001
  99th percentile error: 0.2792
  Mean error: 0.0515
  Max error : 1.6972

--- Noise level: 0% ---
  50th percentile error: 0.0395
  75th percentile error: 0.0556
  90th percentile error: 0.0778
  95th percentile error: 0.1001
  99th percentile error: 0.2792
  Mean error: 0.0515
  Max error : 1.6972

--- Noise level: 1% ---
  50th percentile error: 0.0395
  75th percentile error: 0.0556
  90th percentile error: 0.0778
  95th percentile error: 0.1001
  99th percentile error: 0.2792
  Mean error: 0.0515
  Max error : 1.6972

--- Noise level: 3% ---
  50th percentile error: 0.0395
  75th percentile error: 0.0556
  90th percentile error: 0.0778
  95th percentile error: 0.1001
  99th percentile error: 0.2792
  Mean error: 0.0515
  Max error : 1.6972

--- Noise level: 5% ---
  50th perce

In [8]:
# ============================================================
# Cell 8 — Stress-test evaluation (near-collision, radial extremes,
#          angular degeneracy, OOD)
# ============================================================

def stress_case_near_collision(N=100):
    """Two sources extremely close in rho and phi."""
    X_list, y_list = [], []
    for _ in range(N):
        rho = np.random.uniform(0.2, 0.8, size=3)
        phi = np.random.uniform(-np.pi, np.pi, size=3)

        # Force near-collision between source 1 and 2
        rho[1] = rho[0] + np.random.uniform(-0.01, 0.01)
        phi[1] = phi[0] + np.random.uniform(-0.02, 0.02)

        rho1, phi1, rho2, phi2, rho3, phi3 = canonical_order_three(
            rho[0], phi[0], rho[1], phi[1], rho[2], phi[2]
        )

        feats = get_features_three_sources(rho1, phi1, rho2, phi2, rho3, phi3)
        y_vec = [
            rho1*np.cos(phi1), rho1*np.sin(phi1),
            rho2*np.cos(phi2), rho2*np.sin(phi2),
            rho3*np.cos(phi3), rho3*np.sin(phi3)
        ]

        X_list.append(feats)
        y_list.append(y_vec)

    return np.array(X_list), np.array(y_list)


def stress_case_extreme_radial(N=100):
    """Sources near center or near boundary."""
    X_list, y_list = [], []
    for _ in range(N):
        rho = np.array([
            np.random.uniform(0.02, 0.08),   # near center
            np.random.uniform(0.92, 0.98),   # near boundary
            np.random.uniform(0.4, 0.6)
        ])
        phi = np.random.uniform(-np.pi, np.pi, size=3)

        rho1, phi1, rho2, phi2, rho3, phi3 = canonical_order_three(
            rho[0], phi[0], rho[1], phi[1], rho[2], phi[2]
        )

        feats = get_features_three_sources(rho1, phi1, rho2, phi2, rho3, phi3)
        y_vec = [
            rho1*np.cos(phi1), rho1*np.sin(phi1),
            rho2*np.cos(phi2), rho2*np.sin(phi2),
            rho3*np.cos(phi3), rho3*np.sin(phi3)
        ]

        X_list.append(feats)
        y_list.append(y_vec)

    return np.array(X_list), np.array(y_list)


def stress_case_angular_degeneracy(N=100):
    """Sources nearly collinear in angle."""
    X_list, y_list = [], []
    for _ in range(N):
        base_phi = np.random.uniform(-np.pi, np.pi)
        phi = np.array([
            base_phi,
            base_phi + np.random.uniform(-0.01, 0.01),
            base_phi + np.random.uniform(np.pi - 0.02, np.pi + 0.02)
        ])
        rho = np.random.uniform(0.2, 0.8, size=3)

        rho1, phi1, rho2, phi2, rho3, phi3 = canonical_order_three(
            rho[0], phi[0], rho[1], phi[1], rho[2], phi[2]
        )

        feats = get_features_three_sources(rho1, phi1, rho2, phi2, rho3, phi3)
        y_vec = [
            rho1*np.cos(phi1), rho1*np.sin(phi1),
            rho2*np.cos(phi2), rho2*np.sin(phi2),
            rho3*np.cos(phi3), rho3*np.sin(phi3)
        ]

        X_list.append(feats)
        y_list.append(y_vec)

    return np.array(X_list), np.array(y_list)


def stress_case_OOD(N=100):
    """Out-of-distribution geometry: violates Stage 8 constraints."""
    X_list, y_list = [], []
    for _ in range(N):
        rho = np.random.uniform(0.1, 0.9, size=3)
        phi = np.random.uniform(-np.pi, np.pi, size=3)

        # Force illegal geometry
        rho[1] = rho[0] + np.random.uniform(-0.03, 0.03)
        phi[1] = phi[0] + np.random.uniform(-0.03, 0.03)

        rho1, phi1, rho2, phi2, rho3, phi3 = canonical_order_three(
            rho[0], phi[0], rho[1], phi[1], rho[2], phi[2]
        )

        feats = get_features_three_sources(rho1, phi1, rho2, phi2, rho3, phi3)
        y_vec = [
            rho1*np.cos(phi1), rho1*np.sin(phi1),
            rho2*np.cos(phi2), rho2*np.sin(phi2),
            rho3*np.cos(phi3), rho3*np.sin(phi3)
        ]

        X_list.append(feats)
        y_list.append(y_vec)

    return np.array(X_list), np.array(y_list)


# ------------------------------------------------------------
# Evaluation helper
# ------------------------------------------------------------
def evaluate_stress_case(X_raw, y_raw):
    X_scaled = scaler_X.transform(X_raw)
    X_t = torch.tensor(X_scaled, dtype=torch.float64).to(DEVICE)

    with torch.no_grad():
        rho_p, cos_p, sin_p = model(X_t)

    rho_p = rho_p.cpu().numpy()
    cos_p = cos_p.cpu().numpy()
    sin_p = sin_p.cpu().numpy()

    phi_p = np.arctan2(sin_p, cos_p)
    x_p = rho_p * np.cos(phi_p)
    y_p = rho_p * np.sin(phi_p)

    y_pred = np.column_stack([
        x_p[:,0], y_p[:,0],
        x_p[:,1], y_p[:,1],
        x_p[:,2], y_p[:,2]
    ])

    err = np.linalg.norm(y_pred - y_raw, axis=1)

    # Detect catastrophic failures
    catastrophic = np.sum(err > 0.5)

    # Detect swaps (very rough heuristic)
    swaps = np.sum(np.abs(y_pred[:,0] - y_raw[:,2]) < 0.1)

    return {
        "mean": np.mean(err),
        "p90": np.percentile(err, 90),
        "p95": np.percentile(err, 95),
        "p99": np.percentile(err, 99),
        "max": np.max(err),
        "catastrophic": catastrophic,
        "swaps": swaps
    }


# ------------------------------------------------------------
# Run all stress categories
# ------------------------------------------------------------
categories = {
    "Near Collision": stress_case_near_collision,
    "Extreme Radial": stress_case_extreme_radial,
    "Angular Degeneracy": stress_case_angular_degeneracy,
    "OOD Geometry": stress_case_OOD
}

print("\n================ STRESS TEST RESULTS ================\n")

for name, generator in categories.items():
    print(f"--- {name} ---")
    Xs, ys = generator(100)
    stats = evaluate_stress_case(Xs, ys)
    for k, v in stats.items():
        print(f"  {k}: {v}")
    print()

print("======================================================")



================ STRESS TEST RESULTS ================

--- Near Collision ---
  mean: 0.26786410464924465
  p90: 0.7060073684135554
  p95: 1.4092499815401014
  p99: 1.7475957145276895
  max: 1.8271171975769576
  catastrophic: 14
  swaps: 57

--- Extreme Radial ---
  mean: 0.08408137901704299
  p90: 0.11459137000511031
  p95: 0.12149085410729973
  p99: 0.12792147665181552
  max: 0.1498112929047258
  catastrophic: 0
  swaps: 14

--- Angular Degeneracy ---
  mean: 0.18889281200239832
  p90: 0.45478414677757906
  p95: 1.081171464354
  p99: 1.7080563640991628
  max: 2.18513387682131
  catastrophic: 10
  swaps: 24

--- OOD Geometry ---
  mean: 0.24773344208004158
  p90: 0.5188616478852959
  p95: 0.8280189924392861
  p99: 2.1411783449692843
  max: 2.321464487900664
  catastrophic: 11
  swaps: 59



In [9]:
# ============================================================
# Cell 9 — Triplet-wise error statistics (clean, 3 sources)
# ============================================================

# ------------------------------------------------------------
# 1. Per-source errors
# ------------------------------------------------------------
true_A = y_eval_raw[:, 0:2]
true_B = y_eval_raw[:, 2:4]
true_C = y_eval_raw[:, 4:6]

pred_A = y_pred_cart[:, 0:2]
pred_B = y_pred_cart[:, 2:4]
pred_C = y_pred_cart[:, 4:6]

err_A = np.linalg.norm(pred_A - true_A, axis=1)
err_B = np.linalg.norm(pred_B - true_B, axis=1)
err_C = np.linalg.norm(pred_C - true_C, axis=1)

# ------------------------------------------------------------
# 2. Max error per triplet
# ------------------------------------------------------------
err_max_triplet = np.maximum.reduce([err_A, err_B, err_C])

# ------------------------------------------------------------
# 3. Statistics
# ------------------------------------------------------------
percentiles = [50, 75, 90, 95, 99]

pvals = np.percentile(err_max_triplet, percentiles)

mean_err = np.mean(err_max_triplet)
max_err  = np.max(err_max_triplet)

# Optional: fractions κάτω από thresholds (π.χ. 0.03R, 0.05R, 0.07R, 0.10R)
thr_list = [0.03, 0.05, 0.07, 0.10]
fractions = {thr: np.mean(err_max_triplet < thr) for thr in thr_list}

print("\n=========== TRIPLET-WISE ERROR (CLEAN) ===========\n")

for p, v in zip(percentiles, pvals):
    print(f"{p}th percentile of max-triplet error: {v:.4f}")

print(f"\nMean max-triplet error: {mean_err:.4f}")
print(f"Max  max-triplet error: {max_err:.4f}\n")

print("Fraction of triplets with max error below thresholds:")
for thr, frac in fractions.items():
    print(f"  max_err < {thr:.2f}: {frac*100:.1f}% of samples")

print("\n==================================================")



=========== TRIPLET-WISE ERROR (CLEAN) ===========

50th percentile of max-triplet error: 0.0309
75th percentile of max-triplet error: 0.0441
90th percentile of max-triplet error: 0.0630
95th percentile of max-triplet error: 0.0821
99th percentile of max-triplet error: 0.2230

Mean max-triplet error: 0.0407
Max  max-triplet error: 1.1835

Fraction of triplets with max error below thresholds:
  max_err < 0.03: 47.6% of samples
  max_err < 0.05: 82.0% of samples
  max_err < 0.07: 92.6% of samples
  max_err < 0.10: 96.7% of samples

